In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

In [0]:
import pandas as pd

df = pd.read_csv("/Workspace/Users/smwavali876@gmail.com/Machine_learning/Linear Regression/datasets/insurance.csv")
df

### Explanatory Data Analysis

In [0]:
sns.countplot(data=df, x='sex', hue='smoker')
plt.title("Distribution of Smokers by Gender")
plt.show()

In [0]:
sns.histplot(data=df, x='region', hue='smoker')
plt.title("Distribution of Smokers by Region")
plt.show()

In [0]:
sns.pointplot(df, x='sex', y='charges', hue='smoker')
plt.title("Average Insurance Charges by Gender and Smoker Status")
plt.show()

In [0]:
sns.boxplot(x='smoker', y='charges', data=df, hue='sex')
plt.show()

### Feature Engineering

In [0]:
df.head()

In [0]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df['sex'] = encoder.fit_transform(df['sex'])

In [0]:
df['smoker'] = encoder.fit_transform(df['smoker'])

In [0]:
df.head()

In [0]:
features = df.drop(columns=["charges","region"], axis=1)
features

In [0]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0,10))
scaled_features = scaler.fit_transform(features)
scaled_features

In [0]:
y_features = df["charges"].values
y_features

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(scaled_features, y_features, test_size=0.2)

In [0]:
len(X_train), len(X_test)

In [0]:
X_train

### Linear Regression Model

In [0]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

In [0]:
model.score(X_test, y_test)

In [0]:
from sklearn.metrics import mean_squared_error,r2_score

predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

mse, r2

Predictions

In [0]:
df.head()

In [0]:
samuel_values = scaler.transform([[19, 0, 36.5, 0, 1]])

In [0]:
samuel_values

In [0]:
predictions = model.predict(samuel_values)
predictions

In [0]:
import joblib

joblib.dump(model, 'linear_model.pth')

mlflow

In [0]:
import mlflow
from mlflow.models import infer_signature

with mlflow.start_run():
    pred = model.predict(X_test)
    signature = infer_signature(X_train, pred)
    run_id = mlflow.active_run().info.run_id
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(model, "model", 
                             input_example=X_train[:5],
                             signature=signature
                             )

In [0]:
model_uri = f"runs:/{run_id}/model"
mlflow_model = mlflow.sklearn.load_model(model_uri)

In [0]:
accuracy = mlflow_model.score(X_test, y_test)
accuracy